# NB27: Metal KO λ Landscape and Environmental Metal Contamination Associations

Tests whether the phylogenetic conservatism (Pagel's λ) of a metal-related KO
predicts its association with environmental metal contamination.

**Contamination sources:**
- CSU mobile fractions (PF1, bioavailable fraction; global)
- GeoROC bedrock metals (geological baseline; global)
- USGS raw soil ppm (measured; USA-only)

**Controls:** pH and other env covariates added as second predictor in PGLS.

**Background:** Intercept-only λ for 163 non-metal KOs from same genus set.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial import cKDTree
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT  = Path('/home/hmacgregor/BERIL-research-observatory')
DATA  = ROOT / 'projects/comprehensive_metal_ecology/data'
FIGS  = ROOT / 'projects/comprehensive_metal_ecology/figures'
TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'
USGS_PATH = ROOT / 'projects/metal_contamination_bioindicators/data/usa_samples_usgs.parquet'

sys.path.insert(0, str(ROOT / 'tools'))
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

sys.path.insert(0, str(ROOT / 'projects/comprehensive_metal_ecology/scripts'))
from pgls_utils import load_tree, build_vcv, _optimise_lambda, _gls_fit

print('Setup done.')

Setup done.


In [2]:
# ── Load all static datasets ───────────────────────────────────────────────────
print('Loading data...')

phylo_lam    = pd.read_csv(DATA / 'phylo_d_all_ko.csv')       # 276 metal KOs, unconditional λ
csu_beta     = pd.read_csv(DATA / 'per_ko_lambda_csu_mobility.csv')   # 35×6 CSU PGLS results
env_beta     = pd.read_csv(DATA / 'per_ko_lambda_environmental.csv')  # 35×(GeoROC+NGSA)
genus_csu    = pd.read_csv(DATA / 'genus_csu_mobility.csv')            # genus-level PF1 means
genus_env    = pd.read_csv(DATA / 'genus_lat_env_covariates.csv')      # pH, lat/lon, GeoROC
spark        = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')     # n_genomes, mean_genome_mb
primary      = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')        # 1574 PGLS genera
curated      = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')         # KO metadata

# Presence matrices
nb25      = pd.read_parquet(DATA / 'nb25_ko_presence_matrix.parquet')   # metal KOs (g__ prefix)
all_kos   = pd.read_parquet(DATA / 'genus_ko_presence_all.parquet')     # all KOs (no prefix)

nb25['genus_lower'] = nb25['genus_lower'].str.replace('g__', '', regex=False)

# KO subsets
tier12     = curated[curated['evidence_tier'].isin(['Tier 1', 'Tier 2'])]
FITTED_KOS = sorted(csu_beta['ko_id'].unique())   # 35 KOs that passed n≥30 threshold

# Primary PGLS genera
PGLS_GENERA = set(primary['genus_lower'])

# Non-metal KOs in all_kos
metal_kos    = set(phylo_lam['ko_id'])
nonmetal_kos = set(all_kos['ko'].unique()) - metal_kos

print(f'  Metal KOs (unconditional λ): {len(phylo_lam)}')
print(f'  Fitted Tier1+2 KOs (CSU PGLS): {len(FITTED_KOS)}')
print(f'  Non-metal KOs available: {len(nonmetal_kos)}')
print(f'  Primary PGLS genera: {len(PGLS_GENERA)}')

Loading data...


  Metal KOs (unconditional λ): 276
  Fitted Tier1+2 KOs (CSU PGLS): 35
  Non-metal KOs available: 237
  Primary PGLS genera: 1574


In [3]:
# ── Build metal KO density table (for PGLS predictor) ─────────────────────────
density_base = (nb25[nb25['ko'].isin(FITTED_KOS)]
                .merge(spark[['genus_lower', 'n_genomes', 'mean_genome_mb']], on='genus_lower', how='inner'))
density_base['density']    = density_base['n_genomes_with_ko'] / (density_base['n_genomes'] * density_base['mean_genome_mb'])
density_base['pres_frac']  = density_base['n_genomes_with_ko'] / density_base['n_genomes'].clip(lower=1)

# Background KOs: filter all_kos to primary genera, count eligible KOs
bg_base = (all_kos[all_kos['ko'].isin(nonmetal_kos) & all_kos['genus_lower'].isin(PGLS_GENERA)]
           .merge(spark[['genus_lower', 'n_genomes']], on='genus_lower', how='inner'))
bg_base['pres_frac'] = bg_base['n_genomes_with_ko'] / bg_base['n_genomes'].clip(lower=1)

bg_eligible = (bg_base[bg_base['n_genomes_with_ko'] > 0]
               .groupby('ko')['genus_lower'].nunique())
bg_eligible = bg_eligible[bg_eligible >= 20].index.tolist()

print(f'Background KOs eligible (≥20 genera in PGLS set): {len(bg_eligible)}')
print(f'Metal KO density rows: {len(density_base):,}')

Background KOs eligible (≥20 genera in PGLS set): 163
Metal KO density rows: 43,807


In [4]:
# ── Load tree once ─────────────────────────────────────────────────────────────
print('Loading tree (this takes ~30 s)...')
tree = load_tree(str(TREE_PATH))
tree_labels = {t.label.replace(' ', '_').lower() for t in tree.taxon_namespace}
print(f'  Tree loaded: {len(tree_labels):,} taxa')

def run_pgls_fast(df, response_col, predictor_cols, taxon_col='genus_lower', min_n=30):
    """PGLS using the pre-loaded tree. Returns dict with lambda_est, betas, SEs, p_values, n."""
    keep = df[taxon_col].str.replace(' ', '_').str.lower().isin(tree_labels) & df[response_col].notna()
    for pc in predictor_cols:
        keep = keep & df[pc].notna()
    sub = df[keep].copy()
    sub['_taxon'] = sub[taxon_col].str.replace(' ', '_').str.lower()
    sub = sub.drop_duplicates('_taxon')
    n = len(sub)
    if n < min_n:
        return None
    taxa = sub['_taxon'].tolist()
    y    = sub[response_col].values.astype(float)
    X    = np.column_stack([np.ones(n)] + [sub[pc].values.astype(float) for pc in predictor_cols])
    V    = build_vcv(tree, taxa)
    lam, _ = _optimise_lambda(y, X, V)
    ll, sigma2, betas, betas_se, _, _ = _gls_fit(y, X, V, lam)
    t_stats = betas / np.where(betas_se > 0, betas_se, np.nan)
    df_resid = n - len(betas)
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=df_resid)
    return dict(n=n, lambda_est=lam, betas=betas, SEs=betas_se,
                t_stats=t_stats, p_values=p_values,
                beta=betas[1] if len(betas) > 1 else betas[0],
                SE=betas_se[1] if len(betas_se) > 1 else betas_se[0],
                p_value=p_values[1] if len(p_values) > 1 else p_values[0])

def zscore(x):
    mu, sd = np.nanmean(x), np.nanstd(x, ddof=1)
    return (x - mu) / sd if sd > 0 else x - mu

print('Helper functions ready.')

Loading tree (this takes ~30 s)...
  Tree loaded: 2,283 taxa
Helper functions ready.


## Part A: λ Landscape — Metal KOs by Subcategory

In [5]:
# ── A1: λ by subcategory violin ────────────────────────────────────────────────
CAT_ORDER  = ['Resistance/Detoxification', 'Transport/Homeostasis',
               'Cofactor Biosynthesis', 'Sensing/Regulation',
               'Metal-dependent Metabolism', 'Unknown']
CAT_COLORS = dict(zip(CAT_ORDER, PALETTE))

cat_lams = {}
for cat in CAT_ORDER:
    vals = phylo_lam[phylo_lam['subcategory'] == cat]['lambda'].values
    if len(vals) > 0:
        cat_lams[cat] = vals

print('Subcategory λ summary:')
for cat, lams in cat_lams.items():
    print(f'  {cat:<32} n={len(lams):3d}  median={np.median(lams):.3f}  '
          f'IQR={np.percentile(lams,25):.3f}–{np.percentile(lams,75):.3f}')

if len(cat_lams) >= 3:
    h, p = stats.kruskal(*cat_lams.values())
    print(f'\nKruskal-Wallis across subcategories: H={h:.2f}, p={p:.4e}')

Subcategory λ summary:
  Resistance/Detoxification        n= 59  median=0.526  IQR=0.397–0.720
  Transport/Homeostasis            n= 55  median=0.486  IQR=0.175–0.627
  Cofactor Biosynthesis            n=  4  median=0.547  IQR=0.463–0.630
  Sensing/Regulation               n= 22  median=0.579  IQR=0.286–0.642
  Metal-dependent Metabolism       n= 15  median=0.455  IQR=0.035–0.552
  Unknown                          n=121  median=0.547  IQR=0.456–0.648

Kruskal-Wallis across subcategories: H=8.71, p=1.2121e-01


In [6]:
# ── Figure A1: λ by subcategory — saved now (background added later) ───────────
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(FIGW['full'], ROW_H))
positions, labels = [], []

for cat in CAT_ORDER:
    lams = cat_lams.get(cat)
    if lams is None or len(lams) == 0:
        continue
    pos = len(positions) + 1
    positions.append(pos)
    labels.append(f"{cat.replace('/', '/\\n')}\n(n={len(lams)})")
    col = CAT_COLORS.get(cat, '#7f7f7f')
    if len(lams) >= 3:
        parts = ax.violinplot(lams, positions=[pos], widths=0.65,
                              showmedians=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor(col); pc.set_alpha(0.6)
        for k in ('cmedians', 'cbars', 'cmaxes', 'cmins'):
            parts[k].set_color('black')
    ax.scatter(pos + rng.uniform(-0.15, 0.15, size=len(lams)),
               lams, color=col, s=15, zorder=5, alpha=0.7, edgecolors='none')

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=7)
ax.set_xlabel('KO subcategory', fontsize=9)
ax.set_ylabel("Pagel's λ (intercept-only, gene presence)", fontsize=9)
ax.set_title(f'Metal KO phylogenetic signal by subcategory (n={len(phylo_lam)} KOs)', fontsize=10)
ax.set_ylim(-0.05, 1.15)
grid_h(ax)
fig.suptitle('NB27 Fig A1: λ landscape — metal-associated KOs', y=1.02)
save(fig, FIGS / 'nb27_A1_lambda_by_subcategory')
print('Saved nb27_A1_lambda_by_subcategory.pdf')

Saved nb27_A1_lambda_by_subcategory.pdf


## Part B: Background λ — Non-Metal KOs (163 eligible)

In [7]:
BG_LAMBDA_PATH = DATA / 'nb27_background_lambda.csv'

# Allow partial resume: load existing, skip already-computed KOs
if BG_LAMBDA_PATH.exists():
    bg_results_existing = pd.read_csv(BG_LAMBDA_PATH)
    done_kos = set(bg_results_existing['ko'].tolist())
    bg_rows = bg_results_existing.to_dict('records')
    print(f'Resuming: {len(done_kos)} of {len(bg_eligible)} background KOs already done')
else:
    done_kos = set()
    bg_rows  = []

remaining = [ko for ko in bg_eligible if ko not in done_kos]
print(f'Background λ to compute: {len(remaining)} KOs')
n_total = len(remaining)

for i, ko_id in enumerate(remaining, 1):
    sub = (bg_base[bg_base['ko'] == ko_id]
           .dropna(subset=['pres_frac'])
           .copy())
    taxa = [g.replace(' ', '_').lower() for g in sub['genus_lower']]
    mask = [t in tree_labels for t in taxa]
    taxa_filt = [t for t, ok in zip(taxa, mask) if ok]
    y_arr     = sub['pres_frac'].values[[j for j, ok in enumerate(mask) if ok]].astype(float)

    if len(taxa_filt) < 20:
        continue
    try:
        V = build_vcv(tree, taxa_filt)
        X = np.ones((len(taxa_filt), 1))
        lam, _ = _optimise_lambda(y_arr, X, V)
        bg_rows.append({'ko': ko_id, 'lambda': float(lam), 'n_genera': len(taxa_filt)})
    except Exception as e:
        print(f'  [{i:3d}/{n_total}] ERROR {ko_id}: {e}')
        continue

    if i % 25 == 0 or i == n_total:
        pd.DataFrame(bg_rows).to_csv(BG_LAMBDA_PATH, index=False)
        print(f'  [{i:3d}/{n_total}] checkpoint saved  last: {ko_id} λ={lam:.3f}  n={len(taxa_filt)}')

bg_results = pd.DataFrame(bg_rows)
bg_results.to_csv(BG_LAMBDA_PATH, index=False)
print(f'\nBackground λ complete: {len(bg_results)} KOs saved')
print(f'  median={bg_results["lambda"].median():.3f}  '
      f'IQR={bg_results["lambda"].quantile(0.25):.3f}–{bg_results["lambda"].quantile(0.75):.3f}')

Background λ to compute: 163 KOs


  [ 25/163] checkpoint saved  last: K00764 λ=0.404  n=1481


  [ 50/163] checkpoint saved  last: K01657 λ=0.725  n=1265


  [ 75/163] checkpoint saved  last: K02586 λ=0.645  n=247


  [100/163] checkpoint saved  last: K06989 λ=0.472  n=303


  [125/163] checkpoint saved  last: K14153 λ=0.497  n=102


  [150/163] checkpoint saved  last: K22478 λ=0.294  n=36


  [163/163] checkpoint saved  last: K27545 λ=0.777  n=93

Background λ complete: 163 KOs saved
  median=0.533  IQR=0.421–0.631


In [8]:
# ── Figure B1: Metal KO λ vs background distribution ──────────────────────────
metal_lams_all  = phylo_lam['lambda'].values
bg_lams         = bg_results['lambda'].values

u_stat, p_mw = stats.mannwhitneyu(metal_lams_all, bg_lams, alternative='two-sided')
print(f'Mann-Whitney U: metal vs background λ  U={u_stat:.0f}  p={p_mw:.4e}')
print(f'Metal median: {np.median(metal_lams_all):.3f}  Background median: {np.median(bg_lams):.3f}')

fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Left: KDE density
ax = axs[0]
bins = np.linspace(0, 1, 31)
ax.hist(bg_lams,         bins=bins, density=True, alpha=0.5, color=PALETTE[2],
        edgecolor='k', linewidth=0.5, label=f'Background KOs (n={len(bg_lams)})')
ax.hist(metal_lams_all,  bins=bins, density=True, alpha=0.5, color=PALETTE[0],
        edgecolor='k', linewidth=0.5, label=f'Metal-assoc. KOs (n={len(metal_lams_all)})')
ax.axvline(np.median(metal_lams_all), color=PALETTE[0], lw=1.5, ls='--')
ax.axvline(np.median(bg_lams),        color=PALETTE[2], lw=1.5, ls='--')
ax.annotate(f'Mann-Whitney p={p_mw:.3e}', xy=(0.05, 0.95), xycoords='axes fraction',
            va='top', fontsize=8)
ax.set_xlabel("Pagel's λ", fontsize=9)
ax.set_ylabel('Density', fontsize=9)
ax.set_title('λ distribution: metal vs background KOs', fontsize=10)
ax.legend(fontsize=8)
grid_h(ax)

# Right: violin by subcategory + background
ax2 = axs[1]
groups = {'Background\nnon-metal': bg_lams}
colors = {'Background\nnon-metal': PALETTE[2]}
for cat in CAT_ORDER:
    lams = cat_lams.get(cat)
    if lams is not None and len(lams) >= 3:
        short = cat.split('/')[0]
        groups[short] = lams
        colors[short] = CAT_COLORS.get(cat, '#7f7f7f')

pos_list = list(range(1, len(groups) + 1))
for pos, (lbl, lams) in zip(pos_list, groups.items()):
    col = colors[lbl]
    if len(lams) >= 3:
        parts = ax2.violinplot(lams, positions=[pos], widths=0.6,
                               showmedians=True, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(col); pc.set_alpha(0.6)
        parts['cmedians'].set_color('black')
    ax2.scatter(pos + rng.uniform(-0.12, 0.12, size=len(lams)),
                lams, color=col, s=10, zorder=5, alpha=0.6, edgecolors='none')

ax2.set_xticks(pos_list)
ax2.set_xticklabels([f'{lbl}\n(n={len(lams)})' for lbl, lams in groups.items()], fontsize=7)
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_ylim(-0.05, 1.15)
ax2.set_xlabel('Group', fontsize=9)
ax2.set_ylabel("Pagel's λ", fontsize=9)
ax2.set_title('λ by subcategory + background', fontsize=10)
grid_h(ax2)

fig.suptitle('NB27 Fig B1: Metal KO λ landscape vs background', y=1.02)
save(fig, FIGS / 'nb27_B1_lambda_landscape')
print('Saved nb27_B1_lambda_landscape.pdf')

Mann-Whitney U: metal vs background λ  U=22861  p=7.7536e-01
Metal median: 0.529  Background median: 0.533


Saved nb27_B1_lambda_landscape.pdf


## Part C: λ vs |β_contamination| — CSU and GeoROC (Existing Data)

In [9]:
# ── Join unconditional λ with CSU β values ─────────────────────────────────────
phylo_lam_sub = phylo_lam[phylo_lam['ko_id'].isin(FITTED_KOS)].copy()

# csu_beta already has subcategory/gene_name; only bring in unconditional λ
csu_joined = csu_beta.merge(
    phylo_lam_sub[['ko_id', 'lambda']].rename(columns={'lambda': 'lambda_uncond'}),
    on='ko_id', how='inner'
)
csu_joined['abs_beta'] = np.abs(csu_joined['beta'])
csu_joined['neg_log10_p'] = -np.log10(csu_joined['p_value'].clip(lower=1e-10))
csu_joined['sig'] = csu_joined['p_value'] < 0.05

print('CSU joined shape:', csu_joined.shape)
print('Metals:', sorted(csu_joined['metal'].unique()))
print('\nSpearman ρ (λ_uncond ~ |β_CSU|) per metal:')
for metal, sub in csu_joined.groupby('metal'):
    rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
    print(f'  {metal.upper():3s}: ρ={rho:+.3f}  p={p:.3e}  n={len(sub)}')

CSU joined shape: (210, 13)
Metals: ['As', 'Cd', 'Cr', 'Cu', 'Hg', 'Pb']

Spearman ρ (λ_uncond ~ |β_CSU|) per metal:
  AS : ρ=+0.043  p=8.069e-01  n=35
  CD : ρ=-0.180  p=3.016e-01  n=35
  CR : ρ=-0.381  p=2.395e-02  n=35
  CU : ρ=+0.065  p=7.107e-01  n=35
  HG : ρ=+0.029  p=8.681e-01  n=35
  PB : ρ=-0.303  p=7.659e-02  n=35


In [10]:
# ── Figure C1: λ vs |β_CSU| scatter per metal ─────────────────────────────────
CSU_METALS = sorted(csu_joined['metal'].unique())
n_metals   = len(CSU_METALS)
n_cols     = 3
n_rows     = (n_metals + n_cols - 1) // n_cols

fig, axs = plt.subplots(n_rows, n_cols, figsize=(FIGW['full'], ROW_H * n_rows))
axs_flat = axs.flatten()

for ax, metal in zip(axs_flat, CSU_METALS):
    sub = csu_joined[csu_joined['metal'] == metal]
    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        col = CAT_COLORS.get(cat, '#7f7f7f')
        ax.scatter(ss['lambda_uncond'], ss['abs_beta'], color=col, s=30,
                   alpha=0.8, label=cat.split('/')[0], edgecolors='k', linewidths=0.4)
        # Label significant KOs
        for _, row in ss[ss['sig']].iterrows():
            ax.annotate(row['ko_id'], (row['lambda_uncond'], row['abs_beta']),
                        fontsize=5, alpha=0.7)
    rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
    ax.annotate(f'ρ={rho:+.2f}\np={p:.2e}\nn={len(sub)}',
                xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel("Unconditional Pagel's λ", fontsize=9)
    ax.set_ylabel('|β_CSU_PF1|', fontsize=9)
    ax.set_title(metal.upper(), fontsize=10)
    grid_h(ax)

# Turn off unused panels
for ax in axs_flat[n_metals:]:
    ax.set_visible(False)

# Legend on last visible panel
handles = [plt.scatter([], [], color=CAT_COLORS.get(c, '#7f7f7f'), s=20,
                       label=c.split('/')[0]) for c in CAT_ORDER if c in cat_lams]
axs_flat[n_metals - 1].legend(handles=handles, fontsize=7, loc='lower right')

fig.suptitle('NB27 Fig C1: Unconditional λ vs |β_CSU| — per metal', y=1.02)
save(fig, FIGS / 'nb27_C1_lambda_vs_csu_beta')
print('Saved nb27_C1_lambda_vs_csu_beta.pdf')

Saved nb27_C1_lambda_vs_csu_beta.pdf


In [11]:
# ── Figure C2: λ vs |β_GeoROC| scatter ────────────────────────────────────────
georoc_beta = env_beta[env_beta['source'] == 'GeoROC (bedrock)'].copy()
# env_beta already has subcategory/gene_name; only bring in unconditional λ
georoc_joined = georoc_beta.merge(
    phylo_lam_sub[['ko_id', 'lambda']].rename(columns={'lambda': 'lambda_uncond'}),
    on='ko_id', how='inner'
)
georoc_joined['abs_beta'] = np.abs(georoc_joined['beta'])
georoc_joined['sig'] = georoc_joined['p_value'] < 0.05

GEOROC_METALS = sorted(georoc_joined['metal'].unique())
n_gm = len(GEOROC_METALS)
n_cols2 = 3
n_rows2 = (n_gm + n_cols2 - 1) // n_cols2

fig, axs = plt.subplots(n_rows2, n_cols2, figsize=(FIGW['full'], ROW_H * n_rows2))
axs_flat = axs.flatten() if hasattr(axs, 'flatten') else [axs]

print('GeoROC Spearman ρ (λ_uncond ~ |β_GeoROC|):')
for ax, metal in zip(axs_flat, GEOROC_METALS):
    sub = georoc_joined[georoc_joined['metal'] == metal]
    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['lambda_uncond'], ss['abs_beta'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=30,
                   alpha=0.8, label=cat.split('/')[0],
                   edgecolors='k', linewidths=0.4)
    rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
    print(f'  {metal:3s}: ρ={rho:+.3f}  p={p:.3e}  n={len(sub)}')
    ax.annotate(f'ρ={rho:+.2f}\np={p:.2e}\nn={len(sub)}',
                xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel("Unconditional Pagel's λ", fontsize=9)
    ax.set_ylabel('|β_GeoROC (log bedrock ppm)|', fontsize=9)
    ax.set_title(metal, fontsize=10)
    grid_h(ax)

for ax in axs_flat[n_gm:]:
    ax.set_visible(False)

fig.suptitle('NB27 Fig C2: Unconditional λ vs |β_GeoROC| — per metal', y=1.02)
save(fig, FIGS / 'nb27_C2_lambda_vs_georoc_beta')
print('Saved nb27_C2_lambda_vs_georoc_beta.pdf')

GeoROC Spearman ρ (λ_uncond ~ |β_GeoROC|):
  Co : ρ=-0.448  p=6.988e-03  n=35
  Cr : ρ=+0.288  p=9.393e-02  n=35
  Cu : ρ=-0.243  p=1.593e-01  n=35
  Ni : ρ=-0.218  p=2.085e-01  n=35
  Pb : ρ=-0.264  p=1.256e-01  n=35
  Zn : ρ=-0.066  p=7.047e-01  n=35


Saved nb27_C2_lambda_vs_georoc_beta.pdf


## Part D: USGS Analysis — USA-Only Measured Soil Metal ppm

In [12]:
# ── USGS spatial join: aggregate sample ppm → genus-level means ───────────────
USGS_GENUS_PATH = DATA / 'nb27_genus_usgs_means.csv'

if USGS_GENUS_PATH.exists():
    print('Loading cached USGS genus means...')
    genus_usgs = pd.read_csv(USGS_GENUS_PATH)
else:
    print('Running USGS spatial join...')
    usgs = pd.read_parquet(USGS_PATH)
    USGS_METALS = ['as', 'cd', 'cr', 'cu', 'ni', 'pb', 'zn']
    usgs_raw_cols = [f'usgs_raw_{m}' for m in USGS_METALS]

    # Keep only samples with valid coordinates
    usgs_clean = usgs.dropna(subset=['lat', 'lon']).copy()
    print(f'  USGS samples: {len(usgs_clean):,}')

    # Filter genera to USA bounding box
    USA_LAT = (24.0, 50.0)
    USA_LON = (-126.0, -64.0)
    usa_genera = genus_env[
        genus_env['median_lat'].between(*USA_LAT) &
        genus_env['median_lon'].between(*USA_LON)
    ][['genus_lower', 'median_lat', 'median_lon']].dropna().copy()
    print(f'  Genera in USA bounding box: {len(usa_genera):,}')

    # KD-tree on USGS coordinates
    usgs_coords = usgs_clean[['lat', 'lon']].values
    usgs_tree   = cKDTree(usgs_coords)
    MAX_DIST_DEG = 1.5

    usgs_rows = []
    for _, row in usa_genera.iterrows():
        query_pt = [row['median_lat'], row['median_lon']]
        idxs = usgs_tree.query_ball_point(query_pt, MAX_DIST_DEG)
        if len(idxs) == 0:
            continue
        nearby = usgs_clean.iloc[idxs]
        rec = {'genus_lower': row['genus_lower'], 'n_usgs': len(nearby)}
        for col in usgs_raw_cols:
            vals = nearby[col].dropna()
            rec[f'mean_{col}'] = vals.mean() if len(vals) >= 1 else np.nan
        usgs_rows.append(rec)

    genus_usgs = pd.DataFrame(usgs_rows)
    genus_usgs.to_csv(USGS_GENUS_PATH, index=False)
    print(f'  Genus-USGS join complete: {len(genus_usgs):,} genera with ≥1 USGS sample')

USGS_RAW_COLS = [c for c in genus_usgs.columns if c.startswith('mean_usgs_raw_')]
USGS_METALS_AVAIL = [c.replace('mean_usgs_raw_', '') for c in USGS_RAW_COLS]
print(f'USGS metals available: {USGS_METALS_AVAIL}')
print(f'Genera with USGS data: {len(genus_usgs):,}')

Running USGS spatial join...
  USGS samples: 57,248
  Genera in USA bounding box: 3,700


  Genus-USGS join complete: 3,239 genera with ≥1 USGS sample
USGS metals available: ['as', 'cd', 'cr', 'cu', 'ni', 'pb', 'zn']
Genera with USGS data: 3,239


In [13]:
# ── USGS PGLS: per-KO per-metal, with and without pH control ──────────────────
USGS_PGLS_PATH = DATA / 'nb27_usgs_pgls_results.csv'

if USGS_PGLS_PATH.exists():
    print('Loading cached USGS PGLS results...')
    usgs_pgls = pd.read_csv(USGS_PGLS_PATH)
else:
    print(f'Running USGS PGLS for {len(FITTED_KOS)} KOs × {len(USGS_METALS_AVAIL)} metals × 2 models...')

    # Merge genus USGS means + pH + KO density
    ph_df = genus_env[['genus_lower', 'median_soil_ph']].copy()

    usgs_rows = []
    n_done = 0

    for ko_id in FITTED_KOS:
        density_ko = density_base[density_base['ko'] == ko_id][['genus_lower', 'density']].copy()
        if len(density_ko) == 0:
            continue

        # Join: density × USGS means × pH
        merged = (density_ko
                  .merge(genus_usgs, on='genus_lower', how='inner')
                  .merge(ph_df, on='genus_lower', how='inner'))

        for metal in USGS_METALS_AVAIL:
            metal_col = f'mean_usgs_raw_{metal}'
            sub = merged.dropna(subset=[metal_col, 'density', 'median_soil_ph']).copy()
            if len(sub) < 30:
                continue

            sub['metal_log'] = np.log1p(sub[metal_col])
            sub['density_z'] = zscore(sub['density'])
            sub['ph_z']      = zscore(sub['median_soil_ph'])
            sub['metal_z']   = zscore(sub['metal_log'])

            meta = {
                'ko_id': ko_id,
                'metal': metal,
                'gene_name': tier12[tier12['KO'] == ko_id]['gene_name'].values[0] if ko_id in tier12['KO'].values else '',
                'subcategory': tier12[tier12['KO'] == ko_id]['primary_category'].values[0] if ko_id in tier12['KO'].values else '',
            }

            # Without pH
            res_noph = run_pgls_fast(sub, 'metal_z', ['density_z'])
            if res_noph:
                usgs_rows.append({**meta, 'ph_controlled': False,
                                  'lambda_est': res_noph['lambda_est'],
                                  'beta': res_noph['beta'], 'SE': res_noph['SE'],
                                  'p_value': res_noph['p_value'], 'n_genera': res_noph['n']})

            # With pH
            res_ph = run_pgls_fast(sub, 'metal_z', ['density_z', 'ph_z'])
            if res_ph:
                usgs_rows.append({**meta, 'ph_controlled': True,
                                  'lambda_est': res_ph['lambda_est'],
                                  'beta': res_ph['beta'], 'SE': res_ph['SE'],
                                  'p_value': res_ph['p_value'], 'n_genera': res_ph['n']})

        n_done += 1
        if n_done % 5 == 0:
            print(f'  KO {n_done}/{len(FITTED_KOS)} ({ko_id}) done')

    usgs_pgls = pd.DataFrame(usgs_rows)
    usgs_pgls.to_csv(USGS_PGLS_PATH, index=False)
    print(f'Saved {len(usgs_pgls)} USGS PGLS rows to {USGS_PGLS_PATH}')

print(f'USGS PGLS results: {len(usgs_pgls)} rows')
print(usgs_pgls.groupby(['metal', 'ph_controlled'])['ko_id'].count().unstack())

Running USGS PGLS for 35 KOs × 7 metals × 2 models...


  KO 5/35 (K02190) done


  KO 10/35 (K03523) done


  KO 15/35 (K07785) done


  KO 20/35 (K11602) done


  KO 25/35 (K11708) done


  KO 30/35 (K18367) done
  KO 35/35 (K25287) done
Saved 376 USGS PGLS rows to /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/nb27_usgs_pgls_results.csv
USGS PGLS results: 376 rows
ph_controlled  False  True 
metal                      
as                27     27
cd                26     26
cr                27     27
cu                27     27
ni                27     27
pb                27     27
zn                27     27


In [14]:
# ── Figure D1: λ_uncond vs |β_USGS| per metal (no pH) ────────────────────────
usgs_noph = usgs_pgls[~usgs_pgls['ph_controlled']].copy()
usgs_noph = usgs_noph.merge(
    phylo_lam[['ko_id', 'lambda']].rename(columns={'lambda': 'lambda_uncond'}),
    on='ko_id', how='inner'
)
usgs_noph['abs_beta'] = np.abs(usgs_noph['beta'])
usgs_noph['sig'] = usgs_noph['p_value'] < 0.05

USGS_M_ORDER = sorted(usgs_noph['metal'].unique())
n_um = len(USGS_M_ORDER)
n_cols3 = 4
n_rows3 = (n_um + n_cols3 - 1) // n_cols3

fig, axs = plt.subplots(n_rows3, n_cols3, figsize=(FIGW['full'], ROW_H * n_rows3))
axs_flat = axs.flatten() if hasattr(axs, 'flatten') else [axs]

print('USGS Spearman ρ (λ_uncond ~ |β_USGS|, no pH control):')
for ax, metal in zip(axs_flat, USGS_M_ORDER):
    sub = usgs_noph[usgs_noph['metal'] == metal]
    if len(sub) < 3:
        ax.set_visible(False)
        continue
    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['lambda_uncond'], ss['abs_beta'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=30,
                   alpha=0.8, edgecolors='k', linewidths=0.4)
    if len(sub) >= 3:
        rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
        print(f'  {metal:3s}: ρ={rho:+.3f}  p={p:.3e}  n={len(sub)}')
        ax.annotate(f'ρ={rho:+.2f}\np={p:.2e}\nn={len(sub)}',
                    xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel("Unconditional Pagel's λ", fontsize=9)
    ax.set_ylabel('|β_USGS|', fontsize=9)
    ax.set_title(metal.upper(), fontsize=10)
    grid_h(ax)

for ax in axs_flat[n_um:]:
    ax.set_visible(False)

fig.suptitle('NB27 Fig D1: λ vs |β_USGS| (no pH control) — USA soil ppm', y=1.02)
save(fig, FIGS / 'nb27_D1_lambda_vs_usgs_beta')
print('Saved nb27_D1_lambda_vs_usgs_beta.pdf')

USGS Spearman ρ (λ_uncond ~ |β_USGS|, no pH control):
  as : ρ=+0.092  p=6.496e-01  n=27
  cd : ρ=+0.214  p=2.930e-01  n=26
  cr : ρ=-0.163  p=4.166e-01  n=27
  cu : ρ=+0.380  p=5.034e-02  n=27
  ni : ρ=+0.283  p=1.522e-01  n=27
  pb : ρ=+0.334  p=8.868e-02  n=27
  zn : ρ=+0.206  p=3.018e-01  n=27


Saved nb27_D1_lambda_vs_usgs_beta.pdf


In [15]:
# ── Figure D2: pH control effect on β_USGS ────────────────────────────────────
usgs_ph   = usgs_pgls[usgs_pgls['ph_controlled']].rename(
    columns={'beta': 'beta_ph', 'p_value': 'p_ph', 'n_genera': 'n_genera_ph'})
usgs_comp = usgs_noph.merge(
    usgs_ph[['ko_id', 'metal', 'beta_ph', 'p_ph']],
    on=['ko_id', 'metal'], how='inner')

if len(usgs_comp) > 0:
    fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

    # Left: β no-pH vs β+pH scatter
    ax = axs[0]
    ax.scatter(usgs_comp['beta'], usgs_comp['beta_ph'],
               alpha=0.7, s=20, color=PALETTE[0], edgecolors='k', linewidths=0.4)
    lim = max(np.abs(usgs_comp[['beta', 'beta_ph']].values.flatten()))
    ax.plot([-lim, lim], [-lim, lim], color='gray', lw=0.8, ls='--')
    rho, p = stats.spearmanr(usgs_comp['beta'], usgs_comp['beta_ph'])
    ax.annotate(f'ρ={rho:+.3f}\np={p:.3e}', xy=(0.05, 0.95), xycoords='axes fraction',
                va='top', fontsize=8)
    ax.set_xlabel('β_USGS (no pH control)', fontsize=9)
    ax.set_ylabel('β_USGS (+pH control)', fontsize=9)
    ax.set_title('β attenuation with pH control', fontsize=10)
    grid_h(ax)

    # Right: |Δβ| = |β_noph - β_ph| per metal
    ax2 = axs[1]
    usgs_comp['delta_beta'] = usgs_comp['beta_ph'] - usgs_comp['beta']
    metals_order = sorted(usgs_comp['metal'].unique())
    delta_data = [usgs_comp[usgs_comp['metal'] == m]['delta_beta'].values for m in metals_order]
    ax2.boxplot(delta_data, labels=[m.upper() for m in metals_order],
                patch_artist=True,
                boxprops=dict(facecolor=PALETTE[1], alpha=0.6),
                medianprops=dict(color='k'))
    ax2.axhline(0, color='gray', lw=0.8, ls='--')
    ax2.set_xlabel('Metal', fontsize=9)
    ax2.set_ylabel('Δβ (β_pH − β_no_pH)', fontsize=9)
    ax2.set_title('pH control effect on β_USGS per metal', fontsize=10)
    grid_h(ax2)

    fig.suptitle('NB27 Fig D2: pH control effect on USGS β estimates', y=1.02)
    save(fig, FIGS / 'nb27_D2_usgs_ph_control')
    print('Saved nb27_D2_usgs_ph_control.pdf')
else:
    print('Insufficient USGS pH-controlled data for comparison figure.')

Saved nb27_D2_usgs_ph_control.pdf


## Part E: pH-Controlled CSU PGLS (New Computation)

In [16]:
# ── pH-controlled CSU per-KO PGLS ─────────────────────────────────────────────
CSU_PH_PATH = DATA / 'nb27_csu_ph_pgls_results.csv'

if CSU_PH_PATH.exists():
    print('Loading cached pH-controlled CSU PGLS results...')
    csu_ph_pgls = pd.read_csv(CSU_PH_PATH)
else:
    print('Running pH-controlled CSU PGLS...')

    # Build per-genus CSU + pH table
    csu_env = genus_csu.merge(genus_env[['genus_lower', 'median_soil_ph']], on='genus_lower', how='inner')
    CSU_PF1_METALS = {'As': 'PF1_As', 'Cd': 'PF1_Cd', 'Cr': 'PF1_Cr',
                      'Cu': 'PF1_Cu', 'Hg': 'PF1_Hg', 'Pb': 'PF1_Pb'}

    ph_rows = []
    n_done  = 0

    for ko_id in FITTED_KOS:
        density_ko = density_base[density_base['ko'] == ko_id][['genus_lower', 'density']].copy()
        if len(density_ko) == 0:
            continue

        merged = density_ko.merge(csu_env, on='genus_lower', how='inner')

        meta = {
            'ko_id': ko_id,
            'gene_name': tier12[tier12['KO'] == ko_id]['gene_name'].values[0] if ko_id in tier12['KO'].values else '',
            'subcategory': tier12[tier12['KO'] == ko_id]['primary_category'].values[0] if ko_id in tier12['KO'].values else '',
        }

        for metal, pf1_col in CSU_PF1_METALS.items():
            sub = merged.dropna(subset=[pf1_col, 'density', 'median_soil_ph']).copy()
            if len(sub) < 30:
                continue

            sub['metal_z']   = zscore(sub[pf1_col])
            sub['density_z'] = zscore(sub['density'])
            sub['ph_z']      = zscore(sub['median_soil_ph'])

            # With pH
            res = run_pgls_fast(sub, 'metal_z', ['density_z', 'ph_z'])
            if res:
                ph_rows.append({**meta, 'metal': metal.lower(), 'ph_controlled': True,
                                'lambda_est': res['lambda_est'],
                                'beta': res['beta'], 'SE': res['SE'],
                                'p_value': res['p_value'], 'n_genera': res['n']})

        n_done += 1
        if n_done % 5 == 0:
            print(f'  KO {n_done}/{len(FITTED_KOS)} ({ko_id}) done')

    csu_ph_pgls = pd.DataFrame(ph_rows)
    csu_ph_pgls.to_csv(CSU_PH_PATH, index=False)
    print(f'Saved {len(csu_ph_pgls)} pH-controlled CSU PGLS rows')

print(f'pH-controlled CSU PGLS results: {len(csu_ph_pgls)} rows')

Running pH-controlled CSU PGLS...


  KO 5/35 (K02190) done


  KO 10/35 (K03523) done


  KO 15/35 (K07785) done


  KO 20/35 (K11602) done


  KO 25/35 (K11708) done


  KO 30/35 (K18367) done


  KO 35/35 (K25287) done
Saved 210 pH-controlled CSU PGLS rows
pH-controlled CSU PGLS results: 210 rows


In [ ]:
# ── Figure E1: β_CSU with vs without pH control ────────────────────────────────
# Normalize metal to lowercase before merge (csu_beta has capitalized metals)
csu_orig = (csu_beta[['ko_id', 'metal', 'beta', 'p_value']]
            .assign(metal=lambda d: d['metal'].str.lower())
            .rename(columns={'beta': 'beta_noph', 'p_value': 'p_noph'}))
csu_cmp  = csu_ph_pgls[['ko_id', 'metal', 'subcategory', 'beta', 'p_value']].rename(
    columns={'beta': 'beta_ph', 'p_value': 'p_ph'}).merge(
    csu_orig, on=['ko_id', 'metal'], how='inner')
csu_cmp['delta_beta'] = csu_cmp['beta_ph'] - csu_cmp['beta_noph']

print(f'CSU pH comparison rows: {len(csu_cmp)}')
if len(csu_cmp) > 0:
    rho_cmp, p_cmp = stats.spearmanr(csu_cmp['beta_noph'], csu_cmp['beta_ph'])
    print(f'β no-pH vs β+pH: ρ={rho_cmp:+.3f}, p={p_cmp:.3e}')
    csu_sig_noph = (csu_cmp['p_noph'] < 0.05).sum()
    csu_sig_ph   = (csu_cmp['p_ph'] < 0.05).sum()
    print(f'Significant: no-pH {csu_sig_noph}/{len(csu_cmp)}, +pH {csu_sig_ph}/{len(csu_cmp)}')

    CSU_METALS_CMP = sorted(csu_cmp['metal'].unique())
    fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

    ax = axs[0]
    for cat in CAT_ORDER:
        ss = csu_cmp[csu_cmp['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['beta_noph'], ss['beta_ph'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=20, alpha=0.7,
                   label=cat.split('/')[0], edgecolors='k', linewidths=0.3)
    lim = max(np.abs(csu_cmp[['beta_noph', 'beta_ph']].values.flatten()))
    ax.plot([-lim, lim], [-lim, lim], color='gray', lw=0.8, ls='--')
    ax.annotate(f'ρ={rho_cmp:+.3f}\np={p_cmp:.3e}', xy=(0.05, 0.95), xycoords='axes fraction',
                va='top', fontsize=8)
    ax.set_xlabel('β_CSU_PF1 (no pH control)', fontsize=9)
    ax.set_ylabel('β_CSU_PF1 (+pH control)', fontsize=9)
    ax.set_title('CSU β: no-pH vs pH-controlled', fontsize=10)
    ax.legend(fontsize=7)
    grid_h(ax)

    ax2 = axs[1]
    delta_by_metal = [csu_cmp[csu_cmp['metal'] == m]['delta_beta'].values for m in CSU_METALS_CMP]
    ax2.boxplot(delta_by_metal, tick_labels=[m.upper() for m in CSU_METALS_CMP],
                patch_artist=True,
                boxprops=dict(facecolor=PALETTE[1], alpha=0.6),
                medianprops=dict(color='k'))
    ax2.axhline(0, color='gray', lw=0.8, ls='--')
    ax2.set_xlabel('Metal', fontsize=9)
    ax2.set_ylabel('Δβ (β_pH − β_no_pH)', fontsize=9)
    ax2.set_title('CSU pH control effect per metal', fontsize=10)
    grid_h(ax2)

    fig.suptitle('NB27 Fig E1: pH control effect on CSU β estimates', y=1.02)
    save(fig, FIGS / 'nb27_E1_csu_ph_control')
    print('Saved nb27_E1_csu_ph_control.pdf')
else:
    print('No CSU comparison data available.')

## Summary Statistics

In [18]:
print('=' * 70)
print('NB27 SUMMARY')
print('=' * 70)

print('\n--- λ LANDSCAPE ---')
print(f'Metal KOs (all tiers): n={len(phylo_lam)}')
print(f'  Overall median λ: {phylo_lam["lambda"].median():.3f}')
print(f'  HGT candidates (λ < 0.2): {(phylo_lam["lambda"] < 0.2).sum()}')
print(f'  Vertically inherited (λ > 0.75): {(phylo_lam["lambda"] > 0.75).sum()}')

print(f'\nBackground non-metal KOs: n={len(bg_results)}')
print(f'  Median λ: {bg_results["lambda"].median():.3f}')
u, p_bg = stats.mannwhitneyu(phylo_lam['lambda'], bg_results['lambda'], alternative='two-sided')
print(f'  Mann-Whitney vs metal KOs: p={p_bg:.4e}')

print('\n--- CSU ASSOCIATIONS (no pH) ---')
csu_sig = csu_beta[csu_beta['p_value'] < 0.05]
print(f'Significant KO-metal associations: {len(csu_sig)}/{len(csu_beta)} ({100*len(csu_sig)/len(csu_beta):.1f}%)')
for metal, sub in csu_beta.groupby('metal'):
    sig = (sub['p_value'] < 0.05).sum()
    print(f'  {metal.upper():3s}: {sig}/{len(sub)} significant')

print('\n--- λ vs |β_CSU| CORRELATION (Spearman) ---')
for metal, sub in csu_joined.groupby('metal'):
    rho, p = stats.spearmanr(sub['lambda_uncond'], sub['abs_beta'])
    sig = '*' if p < 0.05 else ''
    print(f'  {metal.upper():3s}: ρ={rho:+.3f}  p={p:.3e} {sig}')

if len(usgs_pgls) > 0:
    print('\n--- USGS ASSOCIATIONS ---')
    usgs_noph_all = usgs_pgls[~usgs_pgls['ph_controlled']]
    usgs_sig = usgs_noph_all[usgs_noph_all['p_value'] < 0.05]
    print(f'KO-metal pairs tested (no pH): {len(usgs_noph_all)}')
    print(f'Significant: {len(usgs_sig)} ({100*len(usgs_sig)/max(len(usgs_noph_all),1):.1f}%)')

print('\nDone.')

NB27 SUMMARY

--- λ LANDSCAPE ---
Metal KOs (all tiers): n=276
  Overall median λ: 0.529
  HGT candidates (λ < 0.2): 47
  Vertically inherited (λ > 0.75): 39

Background non-metal KOs: n=163
  Median λ: 0.533
  Mann-Whitney vs metal KOs: p=7.7536e-01

--- CSU ASSOCIATIONS (no pH) ---
Significant KO-metal associations: 16/210 (7.6%)
  AS : 6/35 significant
  CD : 2/35 significant
  CR : 3/35 significant
  CU : 2/35 significant
  HG : 1/35 significant
  PB : 2/35 significant

--- λ vs |β_CSU| CORRELATION (Spearman) ---
  AS : ρ=+0.043  p=8.069e-01 
  CD : ρ=-0.180  p=3.016e-01 
  CR : ρ=-0.381  p=2.395e-02 *
  CU : ρ=+0.065  p=7.107e-01 
  HG : ρ=+0.029  p=8.681e-01 
  PB : ρ=-0.303  p=7.659e-02 

--- USGS ASSOCIATIONS ---
KO-metal pairs tested (no pH): 188
Significant: 12 (6.4%)

Done.
